# R₀ Calculation for SEIRS-SEI Model

Deriving the basic reproduction number $R_0$ using the Next Generation Matrix (NGM) approach.

## 1. Model Equations

### Human (SEIRS):
$$\frac{dS_H}{dt} = - a b_2 \frac{I_M(t)}{N} S_H + \omega R_H$$
$$\frac{dE_H}{dt} = a b_2 \frac{I_M(t)}{N} S_H - b_{3H} E_H$$
$$\frac{dI_H}{dt} = b_{3H} E_H - \gamma I_H$$
$$\frac{dR_H}{dt} = \gamma I_H - \omega R_H$$

### Mosquito (SEI):
$$\frac{dS_M}{dt} = \Lambda - a b_1 \frac{I_H(t)}{N} S_M - \mu S_M$$
$$\frac{dE_M}{dt} = a b_1 \frac{I_H(t)}{N} S_M - (\mu + b_{3M}\ell) E_M$$
$$\frac{dI_M}{dt} = b_{3M}\ell E_M - \mu I_M$$

<!--where $\ell = e^{-\mu \tau_M}$ is the probability of surviving the sporogonic delay.-->

**Note:** Force of infection uses **current** values for new exposures, progression to infectious uses **delayed** values.

**Mosquito recruitment:**
$$\Lambda = b(R,T) \cdot K \cdot \left(1 - \frac{S_M + E_M + I_M}{K}\right)$$

**Parameters:**
<!-- - $r_H$: Human net growth rate -->
- $a$: Mosquito biting rate
- $b_1$: Probability of transmitting from human to mosquito
- $b_2$: Probability of transmitting from mosquito to human
- $\tau_H$: Human incubation period
- $\tau_M$: Sporogonic delay
- $b_3$: Delay approximation
    - $b_{3H} = \dfrac{1}{\tau_H}$
    - $b_{3M} = \dfrac{T(T-T_{\min\text{Plasm.}})\sqrt{T_{\max\text{Plasm.}}-T}}{DD(T_{\text{opt Plasm.}})\sqrt{T_{\max\text{Plasm.}}-T_{\text{opt Plasm.}}}}$
- $\ell = e^{-\mu \tau_M}$: Probability of surviving sporogony
- $b(R,T)$: Temperature/rain dependent recruitment
- $K$: Mosquito carrying capacity: $K = M \times $ Temperature factor $ \times $ Rain factor $ \times $ Seasonal factor
  - Temperature factor: $\min\left(1, \dfrac{T(T-T_{\min\text{A.}})\sqrt{T_{\max\text{A.}}-T}}{T_{\text{opt A.}}(T_{\text{opt A.}}-T_{\min\text{A.}})\sqrt{T_{\max\text{A.}}-T_{\text{opt A.}}}}\right) $   <!-- $\max\left(0.1, \min\left(1, \dfrac{T - 15}{15}\right)\right)$ -->
  - Rain factor: $\dfrac{2R}{R_L}\exp\left(1 - \dfrac{2R}{R_L} \right)$   <!-- $\min\left(1, \max\left(0, \dfrac{R}{30}\right)\right)$-->
  <!-- - Seasonal factor: $2.5 + 2\sin\left(2\pi\dfrac{t - 180}{365.25}\right) \ (1\leq t \leq 366)$-->
- $\mu$: Mosquito mortality rate
- $\omega$: Immunity loss rate
- $\gamma$: Recovery rate

## 2. Next Generation Matrix Approach

**Infected compartments:** $[E_H, I_H, E_M, I_M]$

Define:
- $\mathcal{F}$: rate of new infections
- $\mathcal{V} = \mathcal{V}^{-} - \mathcal{V}^{+}$: rate of transitions out minus in

At the **Disease-Free Equilibrium (DFE)**:
- $S_H^* = N, E_H^* = I_H^* = R_H^* = 0$
- $S_M^* = \Lambda/\mu, E_M^* = I_M^* = 0 \Longrightarrow S_M^* = b(R,T)\cdot K\cdot\left(1-\frac{S_M^*}{K}\right)/\mu \Longrightarrow S_M^* =\frac{bK}{b+\mu}$

Then $R_0 = \rho(FV^{-1})$

## 3. Computing $\mathcal{F}$ and $\mathcal{V}$
### $\mathcal{F}$ (New Infections)
Only terms that create **newly infected** individuals:
- $\mathcal{F}_1 = a b_2 \frac{I_M}{N} S_H$ → at DFE: $a b_2 I_M$
- $\mathcal{F}_2 = 0$
- $\mathcal{F}_3 = a b_1 \frac{I_H}{N} S_M$ → at DFE: $a b_1 \frac{(\Lambda/\mu)}{N} I_H$
- $\mathcal{F}_4 = 0$
### $\mathcal{V} = \mathcal{V}^{-} - \mathcal{V}^{+}$
Using transition rates $b_{3H} = 1/\tau_H$ and $b_{3M}\ell \approx 1/\tau_M$:
- **For $E_H$ (exposed humans):**
  - $\mathcal{V}_1^{-} = b_{3H} E_H$ (progression to $I_H$)
  - $\mathcal{V}_1^{+} = 0$
  - $\mathcal{V}_1 = b_{3H} E_H$
- **For $I_H$ (infectious humans):**
  - $\mathcal{V}_2^{-} = \gamma I_H$ (recovery)
  - $\mathcal{V}_2^{+} = b_{3H} E_H$ (inflow from $E_H$)
  - $\mathcal{V}_2 = \gamma I_H - b_{3H} E_H$
- **For $E_M$ (exposed mosquitoes):**
  - $\mathcal{V}_3^{-} = b_{3M}\ell E_M + \mu E_M$ (progression to $I_M$ + death)
  - $\mathcal{V}_3^{+} = 0$
  - $\mathcal{V}_3 = (b_{3M}\ell + \mu) E_M$
- **For $I_M$ (infectious mosquitoes):**
  - $\mathcal{V}_4^{-} = \mu I_M$ (death)
  - $\mathcal{V}_4^{+} = b_{3M}\ell E_M$ (inflow from $E_M$)
  - $\mathcal{V}_4 = \mu I_M - b_{3M}\ell E_M$

## 4. Jacobians at DFE
### Computing F from $\mathcal{F}$
$\mathcal{F} = (\mathcal{F}_1, \mathcal{F}_2, \mathcal{F}_3, \mathcal{F}_4)$ 
where:
- $\mathcal{F}_1 = a b_2 \frac{I_M}{N} S_H$
- $\mathcal{F}_3 = a b_1 \frac{I_H}{N} S_M$
Partial derivatives at DFE ($S_H = N, S_M = \Lambda/\mu$):
$$F = \frac{\partial \mathcal{F}}{\partial (E_H, I_H, E_M, I_M)} = \begin{pmatrix} \frac{\partial \mathcal{F}_1}{\partial E_H} & \frac{\partial \mathcal{F}_1}{\partial I_H} & \frac{\partial \mathcal{F}_1}{\partial E_M} & \frac{\partial \mathcal{F}_1}{\partial I_M} \\\frac{\partial \mathcal{F}_2}{\partial E_H} & \frac{\partial \mathcal{F}_2}{\partial I_H} & \frac{\partial \mathcal{F}_2}{\partial E_M} & \frac{\partial \mathcal{F}_2}{\partial I_M} \\\frac{\partial \mathcal{F}_3}{\partial E_H} & \frac{\partial \mathcal{F}_3}{\partial I_H} & \frac{\partial \mathcal{F}_3}{\partial E_M} & \frac{\partial \mathcal{F}_3}{\partial I_M} \\\frac{\partial \mathcal{F}_4}{\partial E_H} & \frac{\partial \mathcal{F}_4}{\partial I_H} & \frac{\partial \mathcal{F}_4}{\partial E_M} & \frac{\partial \mathcal{F}_4}{\partial I_M}\end{pmatrix} = \begin{pmatrix} 0 & 0 & 0 & a b_2 \\0 & 0 & 0 & 0 \\0 & a b_1 \frac{(\Lambda/\mu)}{N} & 0 & 0 \\0 & 0 & 0 & 0\end{pmatrix}$$
### Computing V from $\mathcal{V}$
$\mathcal{V} = (\mathcal{V}_1, \mathcal{V}_2, \mathcal{V}_3, \mathcal{V}_4)$ 
where:
- $\mathcal{V}_1 = b_{3H} E_H$
- $\mathcal{V}_2 = \gamma I_H - b_{3H} E_H$
- $\mathcal{V}_3 = (b_{3M}\ell + \mu) E_M$
- $\mathcal{V}_4 = \mu I_M - b_{3M}\ell E_M$
Partial derivatives
:$$V = \frac{\partial \mathcal{V}}{\partial (E_H, I_H, E_M, I_M)} = \begin{pmatrix} \frac{\partial \mathcal{V}_1}{\partial E_H} & \frac{\partial \mathcal{V}_1}{\partial I_H} & \frac{\partial \mathcal{V}_1}{\partial E_M} & \frac{\partial \mathcal{V}_1}{\partial I_M} \\\frac{\partial \mathcal{V}_2}{\partial E_H} & \frac{\partial \mathcal{V}_2}{\partial I_H} & \frac{\partial \mathcal{V}_2}{\partial E_M} & \frac{\partial \mathcal{V}_2}{\partial I_M} \\\frac{\partial \mathcal{V}_3}{\partial E_H} & \frac{\partial \mathcal{V}_3}{\partial I_H} & \frac{\partial \mathcal{V}_3}{\partial E_M} & \frac{\partial \mathcal{V}_3}{\partial I_M} \\\frac{\partial \mathcal{V}_4}{\partial E_H} & \frac{\partial \mathcal{V}_4}{\partial I_H} & \frac{\partial \mathcal{V}_4}{\partial E_M} & \frac{\partial \mathcal{V}_4}{\partial I_M}\end{pmatrix} = \begin{pmatrix} b_{3H} & 0 & 0 & 0 \\-b_{3H} & \gamma & 0 & 0 \\0 & 0 & b_{3M}\ell + \mu & 0 \\0 & 0 & -b_{3M}\ell & \mu\end{pmatrix}$$

**Determinant:** $\det(V) = b_{3H} \cdot \gamma \cdot \mu \cdot (b_{3M}\ell + \mu) > 0$, so V is **invertible**.

## 5. Next Generation Matrix
$K = F V^{-1}$
**Human block inversion:**
$$V_H = \begin{pmatrix} b_{3H} & 0 \\ -b_{3H} & \gamma \end{pmatrix}, \quad V_H^{-1} = \begin{pmatrix} \frac{1}{b_{3H}} & 0 \\ \frac{1}{\gamma} & \frac{1}{\gamma} \end{pmatrix}$$
**Mosquito block inversion:**
$$V_M = \begin{pmatrix} b_{3M}\ell+\mu & 0 \\ -b_{3M}\ell & \mu \end{pmatrix}, \quad V_M^{-1} = \begin{pmatrix} \frac{1}{b_{3M}\ell+\mu} & 0 \\ \frac{b_{3M}\ell}{(b_{3M}\ell+\mu)\mu} & \frac{1}{\mu} \end{pmatrix}$$
**NGM:**
$$K = F V^{-1} = \begin{pmatrix} 0 & 0 & \frac{a b_2 b_{3M}\ell}{(b_{3M}\ell+\mu)\mu} & \frac{a b_2}{\mu} \\0 & 0 & 0 & 0 \\\frac{a b_1 \Lambda}{N \gamma \mu} & \frac{a b_1 \Lambda\mu}{N \gamma \mu} & 0 & 0 \\0 & 0 & 0 & 0\end{pmatrix}$$
The characteristic polynomial is $\lambda^4 - \left(\frac{a^2 b_1 b_2 \Lambda}{N \gamma \mu} \cdot \frac{b_{3M}\ell}{(b_{3M}\ell+\mu)\mu}\right) \lambda^2 = 0$

The eigenvalues are $0, 0, \pm \sqrt{\frac{a^2 b_1 b_2 \Lambda}{N \gamma \mu} \cdot \frac{b_{3M}\ell}{(b_{3M}\ell+\mu)\mu}}$

## 6. Final R₀ Formula

The spectral radius (dominant eigenvalue) is:

$$R_0 = \frac{a}{\mu}\sqrt{\frac{b_1b_2\Lambda b_{3M}\ell}{N\gamma(b_{3M}\ell+\mu)}}$$

where $\ell = e^{-\mu \tau_M}$ is the probability of surviving the sporogonic delay.

This can be written as:

$$R_0 = \sqrt{R_0^{H} \cdot R_0^{M}}$$

where:
- $R_0^{H} = \frac{a b_2}{\gamma}$ (human side contribution)
- $R_0^{M} = \frac{\Lambda a b_1 b_{3M}\ell}{N (b_{3M}\ell+\mu)\mu^2}$ (mosquito side contribution)


## Summary

| Component | Formula |
|-----------|---------|
**Human (SEIRS)** | $R_0^{H} = \frac{a b_2}{\gamma}$ |
**Mosquito (SEI)** | $R_0^{M} = \frac{\Lambda a b_1}{N\mu} \cdot \frac{b_{3M}\ell}{(b_{3M}\ell+\mu)\mu}$ |
**Combined** | $R_0 = \sqrt{R_0^{H} \cdot R_0^{M}}$ |

**Key insights:**
- $\tau_H$ (human incubation) cancels out (no mortality during this period)
- $\tau_M$ (sporogony) appears via $\ell = e^{-\mu \tau_M}$ (mosquito mortality during incubation reduces transmission)